In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from collections import Counter
from tabulate import tabulate
import random
seed = 42
import os

In [4]:
# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

In [5]:
df = pd.read_csv("data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [6]:
# dataset dimensions
df.shape

(284807, 31)

In [7]:
# dataset information 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [8]:
# dataset description
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [9]:
# check missing values
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [10]:
# check the distribution of classes
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

Dataset is highly imbalanced
0 -> not fraud
1 -> fraud

In [11]:
fraud = df[df.Class == 1]
non_fraud = df[df.Class == 0]
print("no of fraud cases:", fraud.shape[0])
print("no of non-fraud cases:", non_fraud.shape[0])

no of fraud cases: 492
no of non-fraud cases: 284315


In [12]:
# statistical measures of the data
print("Non fraud :")
print(non_fraud.Amount.describe())
print("Fraud :")
print(fraud.Amount.describe())

Non fraud :
count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64
Fraud :
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


In [13]:
# compare values for both classes
df.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


In [14]:
def fitness_function(X, y):
    # Split data into train/test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Instantiate classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Train classifier
    clf.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = clf.predict(X_test)
    
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    
    return acc

In [15]:
def genetic_algorithm_feature_selection(C, y, num_generations=5, population_size=5, mutation_rate=0.2):
    n_features = C.shape[1]
    
    # Initialize population (binary vectors of features)
    population = np.random.randint(2, size=(population_size, n_features))
    print(len(population))
    best_features = None
    best_fitness = 0
    
    for gen in range(num_generations):
        # print(gen)
        fitness_scores = []
        
        # Step 3: Compute fitness for each individual
        for individual in population:
            print(gen,individual)
            selected_features = [i for i in range(n_features) if individual[i] == 1]
            
            # Skip if no feature selected
            if len(selected_features) == 0:
                fitness_scores.append(0)
                continue
            
            acc = fitness_function(C[:, selected_features], y)
            fitness_scores.append(acc)
        
        # Step 4: Optimal fitness
        best_idx = np.argmax(fitness_scores)
        if fitness_scores[best_idx] > best_fitness:
            best_fitness = fitness_scores[best_idx]
            best_features = population[best_idx]
        
        # Step 6: Selection (top 50%)
        sorted_idx = np.argsort(fitness_scores)[::-1]
        population = population[sorted_idx[:population_size // 2]]
        
        # Step 7–9: Crossover and Mutation
        new_population = []
        for _ in range(population_size - len(population)):
            print(_)
            p1, p2 = random.sample(list(population), 2)
            crossover_point = random.randint(1, n_features - 1)
            child = np.concatenate((p1[:crossover_point], p2[crossover_point:]))
            
            # Mutation
            for i in range(n_features):
                if random.random() < mutation_rate:
                    child[i] = 1 - child[i]
            new_population.append(child)
        
        population = np.vstack((population, np.array(new_population)))
        print(f"Generation {gen+1}/{num_generations} | Best Accuracy: {best_fitness:.4f}")
    
    print("Best feature names:", np.where(best_features == 1)[0])
    return best_features, best_fitness


In [16]:
# preparing data 
def prepare_data(data, test_size=0.3, random_state=seed, feature_selection=False, over_sample=False):
    # splitting dataset
    X = data.drop(columns='Class') 
    y = data['Class']
    Xtrain, Xtest, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    print(f"Dimensions after splitting :\n X_train={Xtrain.shape}, X_test={Xtest.shape}, y_train={y_train.shape}, y_test={y_test.shape}")

    # scaling using standard scaler
    scale = StandardScaler()
    X_train = scale.fit_transform(Xtrain)
    X_test = scale.transform(Xtest)
    # print("Before scaling:", Xtrain[:5])
    # print("After scaling:", X_train[:5])

    if feature_selection:
        # Run GA
        selected_features, best_score = genetic_algorithm_feature_selection(X_train, y_train, num_generations=2, population_size=5)
        
        print("Best Accuracy:", best_score)
        print("Selected Feature Indices:", np.where(selected_features == 1)[0])
        X_train= X_train[:, selected_features]
        X_test = X_test[:, selected_features]
    
    if over_sample:
        #oversampling using SMOTE
        print("Before SMOTE:", Counter(y_train))
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print("After SMOTE:", Counter(y_train))
    
    return X_train, X_test, y_train, y_test   

In [17]:
def model_training(X_train, X_test, y_train, y_test):
    models = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "ANN": MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42),
        "Naive Bayes": GaussianNB(),
        "Logistic Regression": LogisticRegression(max_iter=500, random_state=42)
    }
    results = []

    for name, model in models.items():
        print(name)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        auc  = roc_auc_score(y_test, y_pred)
        
        results.append([name, acc, prec, rec, f1, auc])
    return results

In [18]:
def save_results(results, experiment_name):

    metrics_dir = "Results/metrics"
    charts_dir = os.path.join("Results/charts", experiment_name)

    os.makedirs(metrics_dir, exist_ok=True)
    os.makedirs(charts_dir, exist_ok=True)

    results_df = pd.DataFrame(
        results,
        columns=[
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1-score",
            "ROC-AUC"
        ]
    )

    results_df = results_df.sort_values(by="F1-score", ascending=False)

    csv_path = os.path.join(metrics_dir, f"{experiment_name}.csv")
    results_df.to_csv(csv_path, index=False)

    metrics = results_df.columns[1:]

    for metric in metrics:
        plt.figure(figsize=(10,6))

        plt.bar(
            results_df["Model"],
            results_df[metric],
            edgecolor="black"
        )

        plt.title(f"{metric} by Model")
        plt.xticks(rotation=25)

        for i, val in enumerate(results_df[metric]):
            plt.text(i, val+0.01, f"{val:.3f}", ha="center")

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                charts_dir,
                f"{metric.replace('-','_')}.png"
            ),
            dpi=300
        )

        plt.close()

    return results_df

In [19]:
all_results = []
# Prepare data with oversampling on training set
## baseline
X_train, X_test, y_train, y_test = prepare_data(df)
results = model_training(X_train, X_test, y_train, y_test)
results_df = save_results(results, experiment_name="baseline")
results_df["Experiment"] = "Baseline"
all_results.append(results_df)

## with oversampling
X_train, X_test, y_train, y_test = prepare_data(df, over_sample=True)
results = model_training(X_train, X_test, y_train, y_test)
results_df = save_results(results, experiment_name="oversampling")
results_df["Experiment"] = "Oversampling"
all_results.append(results_df)

## with feature selection and oversampling with 5 set of feature vectors
for i in range(5):
    print(f"Experiment {i+1}/5")
    X_train, X_test, y_train, y_test = prepare_data(df, feature_selection=True, over_sample=True)
    results = model_training(X_train, X_test, y_train, y_test)
    results_df = save_results(results, experiment_name=f"feature_selection_oversampling_{i+1}")
    results_df["Experiment"] = f"Feature Selection_set {i+1}"
    all_results.append(results_df)
   
    
combined_results = pd.concat(all_results, ignore_index=True)    
print(tabulate(
    combined_results,
    headers="keys",
    tablefmt="github",
    showindex=False
))

Dimensions after splitting :
 X_train=(199364, 30), X_test=(85443, 30), y_train=(199364,), y_test=(85443,)
Random Forest
Decision Tree
ANN
Naive Bayes
Logistic Regression
Dimensions after splitting :
 X_train=(199364, 30), X_test=(85443, 30), y_train=(199364,), y_test=(85443,)
Before SMOTE: Counter({0: 199020, 1: 344})
After SMOTE: Counter({0: 199020, 1: 199020})
Random Forest
Decision Tree
ANN
Naive Bayes
Logistic Regression
Experiment 1/5
Dimensions after splitting :
 X_train=(199364, 30), X_test=(85443, 30), y_train=(199364,), y_test=(85443,)
5
0 [1 0 1 0 1 1 1 1 0 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 1 1 1 0 0 1]
0 [1 1 0 1 1 1 0 0 1 1 1 0 1 1 1 0 1 0 1 1 1 0 0 0 0 0 1 1 0 0]
0 [0 0 1 1 1 1 0 1 1 0 1 1 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 0 0 0]
0 [0 1 0 1 1 0 1 1 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 1 0 1 1 0 0 1]
0 [0 1 0 0 0 1 1 0 0 0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 0 0 1 1 1 0]
0
1
2
Generation 1/2 | Best Accuracy: 0.9996
1 [1 0 1 0 1 1 1 1 0 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 1 1 1 0 0 1]
1 [0 1 0 0 0 1 1 

In [20]:
print(tabulate(
    combined_results,
    headers="keys",
    tablefmt="latex",
    showindex=False
))

\begin{tabular}{lrrrrrl}
\hline
 Model               &   Accuracy &   Precision &   Recall &   F1-score &   ROC-AUC & Experiment              \\
\hline
 Random Forest       &   0.99952  &  0.957265   & 0.756757 &  0.845283  &  0.878349 & Baseline                \\
 ANN                 &   0.999462 &  0.939655   & 0.736486 &  0.825758  &  0.868202 & Baseline                \\
 Decision Tree       &   0.999181 &  0.778571   & 0.736486 &  0.756944  &  0.868062 & Baseline                \\
 Logistic Regression &   0.999146 &  0.850467   & 0.614865 &  0.713725  &  0.807339 & Baseline                \\
 Naive Bayes         &   0.978009 &  0.0604368  & 0.804054 &  0.112423  &  0.891182 & Baseline                \\
 Random Forest       &   0.999462 &  0.892308   & 0.783784 &  0.834532  &  0.89181  & Oversampling            \\
 ANN                 &   0.999064 &  0.717949   & 0.756757 &  0.736842  &  0.87812  & Oversampling            \\
 Decision Tree       &   0.997097 &  0.340764   & 0.72297